# Section 1 — Environment Setup

In [1]:

import sys
import os
import platform
import importlib.metadata as metadata

print("=== Python Environment ===")
print("Python version:", sys.version)
print("Platform:", platform.platform())

packages_to_check = [
    "pandas",
    "numpy",
    "scikit-learn",
    "pyarrow",
]

print("\n=== Required Data Packages ===")

for package in packages_to_check:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


=== Python Environment ===
Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Platform: Linux-6.18.48+-x86_64-with-glibc2.39

=== Required Data Packages ===
pandas: 2.3.3
numpy: 2.1.3
scikit-learn: 1.6.1
pyarrow: 23.0.1



# Section 2: Dataset Acquisition and File Discovery

## Objective
Locate the MMLoSo 2025 dataset and identify the raw files used for the four target languages: Bhili, Gondi, Mundari, and Santali.

## Dataset configuration
- Bhili: Hindi → Bhili
- Gondi: Hindi → Gondi
- Mundari: Hindi → Mundari
- Santali: English → Santali

## Reproducibility note
Raw data will be preserved separately from processed data. The source and target sentences must remain aligned throughout preprocessing.


In [5]:
from pathlib import Path
import os

# Prefer the competition dataset already attached to this notebook.
DATA_DIR = Path("/kaggle/input/competitions/mmloso2025")

if DATA_DIR.exists() and (DATA_DIR / "santali-train.csv").exists():
    print("Using the attached Kaggle competition dataset:")
    print(DATA_DIR)

else:
    # Download through KaggleHub if the attached input is unavailable.
    import kagglehub

    print("Competition input not found.")
    print("Downloading the MMLoSO 2025 dataset using KaggleHub...")

    # The competition slug must be accessible to your Kaggle account.
    DATA_DIR = Path(
        kagglehub.competition_download("mmloso2025")
    )

    print("Downloaded dataset location:", DATA_DIR)

# Validate the files needed by our pipeline.
required_files = [
    "bhili-train.csv",
    "gondi-train.csv",
    "mundari-train.csv",
    "santali-train.csv",
    "test.csv",
]

missing_files = [
    filename
    for filename in required_files
    if not (DATA_DIR / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing required dataset files: {missing_files}\n"
        f"Dataset directory checked: {DATA_DIR}"
    )

print("\nAll required dataset files are available.")

Using the attached Kaggle competition dataset:
/kaggle/input/competitions/mmloso2025

All required dataset files are available.


In [6]:
# Load the four parallel training datasets
bhili = pd.read_csv(DATA_DIR / "bhili-train.csv")
gondi = pd.read_csv(DATA_DIR / "gondi-train.csv")
mundari = pd.read_csv(DATA_DIR / "mundari-train.csv")
santali = pd.read_csv(DATA_DIR / "santali-train.csv")

datasets_raw = {
    "bhili": bhili,
    "gondi": gondi,
    "mundari": mundari,
    "santali": santali,
}

print("=== Loaded Training Datasets ===")

for language, df in datasets_raw.items():
    print(
        f"{language.capitalize():10s}: "
        f"{len(df):,} rows, {len(df.columns)} columns"
    )

print("\nSource and target columns:")
for language, df in datasets_raw.items():
    target_column = {
        "bhili": "Bhili",
        "gondi": "Gondi",
        "mundari": "Mundari",
        "santali": "Santali",
    }[language]

    source_column = "English" if language == "santali" else "Hindi"
    print(f"{language.capitalize():10s}: {source_column} -> {target_column}")

print("\nOfficial test.csv remains separate.")

=== Loaded Training Datasets ===
Bhili     : 20,000 rows, 3 columns
Gondi     : 20,000 rows, 3 columns
Mundari   : 20,000 rows, 3 columns
Santali   : 20,000 rows, 3 columns

Source and target columns:
Bhili     : Hindi -> Bhili
Gondi     : Hindi -> Gondi
Mundari   : Hindi -> Mundari
Santali   : English -> Santali

Official test.csv remains separate.


## Section 3 — Data Cleaning

This section applies consistent preprocessing rules to all four parallel datasets.

Cleaning operations:
1. Select only the source and target sentence columns.
2. Remove rows with missing source or target sentences.
3. Convert sentence values to strings and strip leading/trailing whitespace.
4. Remove empty source or target sentences.
5. Remove duplicate source–target pairs.
6. Reset the row index.

The original `Unnamed: 0` identifier is excluded because it is not a translation input or target. The original dataframes remain unchanged.

In [7]:
# Define source and target columns for each language
language_columns = {
    "bhili": ("Hindi", "Bhili"),
    "gondi": ("Hindi", "Gondi"),
    "mundari": ("Hindi", "Mundari"),
    "santali": ("English", "Santali"),
}

def clean_parallel_dataset(df, source_col, target_col):
    cleaned_df = df[[source_col, target_col]].copy()

    # Remove missing sentence pairs
    cleaned_df = cleaned_df.dropna(
        subset=[source_col, target_col]
    )

    # Normalize whitespace
    for col in [source_col, target_col]:
        cleaned_df[col] = cleaned_df[col].astype(str).str.strip()

    # Remove empty sentences
    cleaned_df = cleaned_df[
        (cleaned_df[source_col] != "") &
        (cleaned_df[target_col] != "")
    ]

    # Remove duplicate parallel pairs
    cleaned_df = cleaned_df.drop_duplicates(
        subset=[source_col, target_col]
    )

    return cleaned_df.reset_index(drop=True)


# Clean all four datasets consistently
cleaned = {}

for language, df in datasets_raw.items():
    source_col, target_col = language_columns[language]

    cleaned[language] = clean_parallel_dataset(
        df,
        source_col,
        target_col
    )

print("=== Cleaning Summary ===")

for language in datasets_raw:
    original_rows = len(datasets_raw[language])
    cleaned_rows = len(cleaned[language])
    removed_rows = original_rows - cleaned_rows

    print(
        f"{language.capitalize():10s}: "
        f"{original_rows:,} -> {cleaned_rows:,} rows "
        f"({removed_rows:,} removed)"
    )

print("\n=== Sample Cleaned Santali Pairs ===")
display(cleaned["santali"].head(5))

=== Cleaning Summary ===
Bhili     : 20,000 -> 19,802 rows (198 removed)
Gondi     : 20,000 -> 20,000 rows (0 removed)
Mundari   : 20,000 -> 20,000 rows (0 removed)
Santali   : 20,000 -> 19,870 rows (130 removed)

=== Sample Cleaned Santali Pairs ===


,English,Santali
0,"Deno 1.0 was released on May 13, 2020.","ᱰᱮᱱᱳ ᱑ᱹ᱐ ᱫᱚ ᱑᱓ ᱢᱮ, ᱒᱐᱒᱐ ᱨᱮ ᱪᱷᱟᱹᱲᱮ ᱧᱟᱢ ᱞᱮᱫᱟ ᱾"
1,Denuvo Software Solutions GmbH is a subsidary ...,ᱰᱮᱱᱩᱵᱷᱳ ᱥᱚᱯᱷᱴᱳᱣᱮᱨ ᱥᱚᱞᱤᱣᱥᱚᱱᱥ ᱡᱤ ᱮᱢ ᱵᱤ ᱮᱭᱤᱪ ᱫᱚ ᱤ...
2,Many books specialise in the details of partic...,ᱟᱭᱢᱟ ᱯᱚᱛᱚᱵ ᱠᱚᱜᱮ ᱵᱤᱥᱟᱹᱥ ᱥᱚᱯᱷᱴᱳᱣᱮᱨ ᱨᱮᱭᱟᱜ ᱵᱚᱨᱱᱚᱱ ...
3,Deterministic algorithm A deterministic algor...,ᱰᱤᱴᱟᱨᱢᱤᱱᱤᱥᱴᱤᱠ ᱮᱞᱜᱚᱨᱤᱛᱷᱚᱢ ᱰᱤᱴᱟᱨᱢᱤᱱᱤᱥᱴᱤᱠ ᱮᱞᱜᱚᱨᱤᱛ...
4,It comes with a C compiler and an assembler.,ᱱᱚᱣᱟ ᱫᱚ ᱢᱤᱫᱴᱟᱝ ᱥᱤ ᱠᱚᱢᱯᱟᱭᱞᱟᱨ ᱟᱨ ᱢᱤᱫᱴᱟᱝ ᱮᱥᱮᱢᱵᱞᱟᱨ...


In [11]:
# Comprehensive post-cleaning validation

validation_results = []

for language, df in cleaned.items():
    source_col, target_col = language_columns[language]

    # Missing values
    missing_source = int(df[source_col].isna().sum())
    missing_target = int(df[target_col].isna().sum())

    # Empty or whitespace-only strings
    empty_source = int(
        df[source_col].astype(str).str.strip().eq("").sum()
    )
    empty_target = int(
        df[target_col].astype(str).str.strip().eq("").sum()
    )

    # Leading/trailing whitespace remaining
    source_whitespace = int(
        df[source_col].ne(df[source_col].astype(str).str.strip()).sum()
    )
    target_whitespace = int(
        df[target_col].ne(df[target_col].astype(str).str.strip()).sum()
    )

    # Exact duplicate parallel pairs
    duplicate_pairs = int(
        df.duplicated(subset=[source_col, target_col]).sum()
    )

    # Check for non-string values
    non_string_source = int(
        (~df[source_col].map(lambda x: isinstance(x, str))).sum()
    )
    non_string_target = int(
        (~df[target_col].map(lambda x: isinstance(x, str))).sum()
    )

    validation_results.append({
        "language": language,
        "rows": len(df),
        "missing_source": missing_source,
        "missing_target": missing_target,
        "empty_source": empty_source,
        "empty_target": empty_target,
        "source_whitespace": source_whitespace,
        "target_whitespace": target_whitespace,
        "duplicate_pairs": duplicate_pairs,
        "non_string_source": non_string_source,
        "non_string_target": non_string_target,
    })

cleaning_validation_df = pd.DataFrame(validation_results)

display(cleaning_validation_df)

# Fail if any required cleaning condition is violated
checks_to_pass = [
    "missing_source",
    "missing_target",
    "empty_source",
    "empty_target",
    "source_whitespace",
    "target_whitespace",
    "duplicate_pairs",
    "non_string_source",
    "non_string_target",
]

all_passed = cleaning_validation_df[checks_to_pass].eq(0).all().all()

print("\nAll cleaning checks passed:", all_passed)

assert all_passed, (
    "Cleaning validation failed. Inspect the report before proceeding."
)

,language,rows,missing_source,missing_target,empty_source,empty_target,source_whitespace,target_whitespace,duplicate_pairs,non_string_source,non_string_target
0,bhili,19802,0,0,0,0,0,0,0,0,0
1,gondi,20000,0,0,0,0,0,0,0,0,0
2,mundari,20000,0,0,0,0,0,0,0,0,0
3,santali,19870,0,0,0,0,0,0,0,0,0



All cleaning checks passed: True


## Section 4 — Duplicate and Data-Leakage Checks

This section checks the cleaned parallel datasets before splitting.

We will measure:
- Duplicate source–target pairs.
- Repeated source sentences.
- Repeated target sentences.
- Source sentences associated with multiple distinct target translations.

These checks help us understand dataset quality and identify potential leakage risks. We will not remove every repeated source or target automatically, because legitimate translations can share common sentences or expressions.

In [8]:
# Check duplicate patterns in each cleaned dataset

duplicate_report = []

for language, df in cleaned.items():
    source_col, target_col = language_columns[language]

    duplicate_pairs = int(
        df.duplicated(subset=[source_col, target_col]).sum()
    )

    repeated_sources = int(
        df[source_col].duplicated(keep=False).sum()
    )

    repeated_targets = int(
        df[target_col].duplicated(keep=False).sum()
    )

    # Number of distinct source sentences linked to multiple targets
    targets_per_source = df.groupby(source_col)[target_col].nunique()
    sources_with_multiple_targets = int(
        (targets_per_source > 1).sum()
    )

    duplicate_report.append({
        "language": language,
        "rows": len(df),
        "duplicate_pairs": duplicate_pairs,
        "rows_with_repeated_source": repeated_sources,
        "rows_with_repeated_target": repeated_targets,
        "sources_with_multiple_targets": sources_with_multiple_targets,
    })

duplicate_report_df = pd.DataFrame(duplicate_report)

display(duplicate_report_df)

,language,rows,duplicate_pairs,rows_with_repeated_source,rows_with_repeated_target,sources_with_multiple_targets
0,bhili,19802,0,472,36,231
1,gondi,20000,0,0,215,0
2,mundari,20000,0,96,26,41
3,santali,19870,0,2412,73,1184


## Section 5 — Reproducible Train, Validation, and Test Splits

Each cleaned language dataset is divided into:
- 90% training data
- 5% validation data
- 5% internal test data

A fixed random seed ensures reproducibility. Split assignments will be saved so all experiments can use identical examples.

The validation set is used for model selection and early stopping. The internal test set is reserved for final evaluation. The official competition test file remains separate.

In [9]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
TRAIN_RATIO = 0.90
VAL_RATIO = 0.05
TEST_RATIO = 0.05

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

splits = {}

for language, df in cleaned.items():
    # First separate 10% for validation + internal test
    train_df, remaining_df = train_test_split(
        df,
        test_size=VAL_RATIO + TEST_RATIO,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    # Divide the remaining 10% equally into validation and test
    val_df, test_df = train_test_split(
        remaining_df,
        test_size=0.5,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    splits[language] = {
        "train": train_df.reset_index(drop=True),
        "validation": val_df.reset_index(drop=True),
        "test": test_df.reset_index(drop=True),
    }

print("=== Split Summary ===")

for language, parts in splits.items():
    total = sum(len(part) for part in parts.values())

    print(f"\n{language.capitalize()}: {total:,} total rows")

    for split_name, split_df in parts.items():
        percentage = len(split_df) / total * 100
        print(
            f"  {split_name:10s}: "
            f"{len(split_df):,} rows ({percentage:.2f}%)"
        )

=== Split Summary ===

Bhili: 19,802 total rows
  train     : 17,821 rows (90.00%)
  validation: 990 rows (5.00%)
  test      : 991 rows (5.00%)

Gondi: 20,000 total rows
  train     : 18,000 rows (90.00%)
  validation: 1,000 rows (5.00%)
  test      : 1,000 rows (5.00%)

Mundari: 20,000 total rows
  train     : 18,000 rows (90.00%)
  validation: 1,000 rows (5.00%)
  test      : 1,000 rows (5.00%)

Santali: 19,870 total rows
  train     : 17,883 rows (90.00%)
  validation: 993 rows (5.00%)
  test      : 994 rows (5.00%)


In [10]:
# Verify that train, validation, and test contain no overlapping pairs.

print("=== Split Integrity Checks ===")

for language, parts in splits.items():
    source_col, target_col = language_columns[language]

    train_pairs = set(
        zip(parts["train"][source_col], parts["train"][target_col])
    )
    val_pairs = set(
        zip(parts["validation"][source_col], parts["validation"][target_col])
    )
    test_pairs = set(
        zip(parts["test"][source_col], parts["test"][target_col])
    )

    train_val_overlap = len(train_pairs & val_pairs)
    train_test_overlap = len(train_pairs & test_pairs)
    val_test_overlap = len(val_pairs & test_pairs)

    total_split_rows = sum(len(part) for part in parts.values())

    print(f"\n{language.capitalize()}")
    print("Train–validation overlapping pairs:", train_val_overlap)
    print("Train–test overlapping pairs:", train_test_overlap)
    print("Validation–test overlapping pairs:", val_test_overlap)
    print("All rows accounted for:",
          total_split_rows == len(cleaned[language]))

    assert train_val_overlap == 0
    assert train_test_overlap == 0
    assert val_test_overlap == 0
    assert total_split_rows == len(cleaned[language])

print("\nAll split integrity checks passed.")

=== Split Integrity Checks ===

Bhili
Train–validation overlapping pairs: 0
Train–test overlapping pairs: 0
Validation–test overlapping pairs: 0
All rows accounted for: True

Gondi
Train–validation overlapping pairs: 0
Train–test overlapping pairs: 0
Validation–test overlapping pairs: 0
All rows accounted for: True

Mundari
Train–validation overlapping pairs: 0
Train–test overlapping pairs: 0
Validation–test overlapping pairs: 0
All rows accounted for: True

Santali
Train–validation overlapping pairs: 0
Train–test overlapping pairs: 0
Validation–test overlapping pairs: 0
All rows accounted for: True

All split integrity checks passed.


## Section 6 — Save Shared Dataset Artifacts

This section saves the validated datasets and their fixed train, validation, and internal test splits.

For reproducibility, we will save:
- CSV and Parquet files for each split.
- A shared experiment configuration containing the base model, split ratios, and random seed.
- A dataset manifest containing row counts, column mappings, and file hashes.

These artifacts will be stored in `/kaggle/working/santali_common_preprocessing`. Because the working directory is temporary, the completed artifacts must be downloaded or published as a Kaggle Dataset before they can reliably be reused by other notebooks.

In [13]:
import json
import hashlib
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/common-tribal-preprocessing")
SPLIT_DIR = OUTPUT_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "ai4bharat/indictrans2-indic-en-1B"

experiment_config = {
    "base_model": MODEL_NAME,
    "random_seed": RANDOM_SEED,
    "split_ratios": {
        "train": TRAIN_RATIO,
        "validation": VAL_RATIO,
        "test": TEST_RATIO,
    },
    "languages": language_columns,
    "official_competition_test_used_for_splitting": False,
}

manifest = {
    "dataset_source": "MMLoSO 2025 competition training CSVs",
    "cleaning_validation_passed": True,
    "cleaning_operations": [
        "drop missing source-target pairs",
        "strip leading and trailing whitespace",
        "remove empty sentences",
        "remove duplicate source-target pairs",
    ],
    "languages": {},
}

for language, parts in splits.items():
    source_col, target_col = language_columns[language]

    manifest["languages"][language] = {
        "source_column": source_col,
        "target_column": target_col,
        "cleaned_rows": len(cleaned[language]),
        "splits": {},
    }

    for split_name, split_df in parts.items():
        filename = f"{language}_{split_name}"
        csv_path = SPLIT_DIR / f"{filename}.csv"
        parquet_path = SPLIT_DIR / f"{filename}.parquet"

        split_df.to_csv(csv_path, index=False, encoding="utf-8")
        split_df.to_parquet(parquet_path, index=False)

        csv_hash = hashlib.sha256(
            csv_path.read_bytes()
        ).hexdigest()

        manifest["languages"][language]["splits"][split_name] = {
            "rows": len(split_df),
            "csv_file": str(csv_path.relative_to(OUTPUT_DIR)),
            "parquet_file": str(parquet_path.relative_to(OUTPUT_DIR)),
            "csv_sha256": csv_hash,
        }

(OUTPUT_DIR / "experiment_config.json").write_text(
    json.dumps(experiment_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

(OUTPUT_DIR / "dataset_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Artifacts saved to:", OUTPUT_DIR)
print("Languages:", len(splits))
print("Split datasets:", sum(len(parts) for parts in splits.values()))
print("CSV files:", sum(len(parts) for parts in splits.values()))
print("Parquet files:", sum(len(parts) for parts in splits.values()))
print("Configuration and manifest saved.")

Artifacts saved to: /kaggle/working/common-tribal-preprocessing
Languages: 4
Split datasets: 12
CSV files: 12
Parquet files: 12
Configuration and manifest saved.


In [14]:
import shutil
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/common-tribal-preprocessing")
ZIP_PATH = Path("/kaggle/working/common-tribal-preprocessing.zip")

shutil.make_archive(
    str(ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=OUTPUT_DIR
)

print("ZIP created:", ZIP_PATH)
print("Contains all four languages and their splits.")

ZIP created: /kaggle/working/common-tribal-preprocessing.zip
Contains all four languages and their splits.
